In [9]:
import asyncio
from agents import Agent, FileSearchTool, Runner, WebSearchTool
import os
from dotenv import load_dotenv

from IPython.display import display, Markdown
import nest_asyncio

import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field
from sqlalchemy.util import await_only

from src.Agent_OCR import DocumentOCR, load_from_json
from src.Agent_User_Profile import *
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("mck_openai_api_key") or os.getenv("OPENAI_API_KEY", "")
os.environ["OPENAI_BASE_URL"] = os.getenv("mck_openai_base_url") or os.getenv("OPENAI_BASE_URL", "")

## Load Data

In [10]:
message =load_from_json("../Data/Intermediate/Client_Input_OCR/W2_XL_input_clean_2999.json")
print(len(message))

4385


In [11]:
complete_user_profile = {'Social_Security_Number': '522-86-4190', 'Name': 'Daniel Robinson', 'Email': 'Francisco_Reveriano@zoho.com', 'filingStatus': 'Single', 'dependents': 0, 'address': '8432 Castro Well West Anna NJ 73837-7432', 'state': 'NJ', 'Occupation': 'Consultant', 'spouse': SpouseProfile(name='', occupation='', birthYear=0), 'complete': True, 'complete_reasoning': 'The user supplied their email address, confirmed their filing status as Single, stated they have 0 dependents, and indicated their occupation is Consultant. Since these were the only missing pieces of information, and no spouse details are required for a Single filer, the profile is now complete.', 'missing_questions': []}
print(len(str(dict(complete_user_profile))))

678


In [12]:
updated_message = "# Full User Profile\n" + str(dict(complete_user_profile)) + "\n\n" + "# Employee W-2\n" + message
print(len(updated_message))

5100


## Define Agent

In [13]:
class W2Profile(BaseModel):
    TaxYear : int
    "Tax Year from Employee W-2"
    Employer: str
    "Employer Name from the Employee W-2"
    Employer_EIN: str
    "Employer Identification Number (EIN) from the Employee W-2"
    Employee_Name: str
    "Employee Name from the Employee W-2"
    Employee_SSN: str
    "Employee Social Security Number from the Employee W-2"
    Employee_DOB: str
    "Employee Date of Birth from the Employee W-2"
    Employee_DoB: str
    "Employee Date of Birth from the Employee W-2"
    Wages_Income: float
    "Employee Wages and Income from the Employee W-2"
    Federal_Taxes_Withheld: float
    "Employee Federal Taxes Withheld from the Employee W-2"
    Filling_Status: Literal['Single', "Married Filing Jointly", "Married Filing Separately", "Head of Household", "Qualifying Widow(er)"]
    "Employee Filing Status from the Employee W-2"
    State_Information: str
    "Employee State Information (e.g., States and wages withold by the state)  from the Employee W-2"
    Questions: str
    "Any questions about the employee from the Employee W-2? Or information that is not clear"
    Score: Literal["High", "Medium", "Low"]
    "Score for the user profile based on the questions and answers"

In [18]:
PROMPT = '''
You are a helpful tax agent. You thoroughly read the document and provide the required information.
'''

W2_Profile_Agent = Agent(
        name="W2ProfileAgent",
        instructions=PROMPT,
        model="o3",
        output_type=W2Profile,
        )
W2_Profile_Table_Agent = Agent(
    name="W2ProfileTableAgent",
        instructions="Create well constructed Markdown table from the provided user profile. Skip 'Questions' and 'Score'. Double-check the response and make sure you return nothing but the table.",
        model="gpt-4.1",
        output_type=str,
)

In [19]:
result = await Runner.run(
            W2_Profile_Agent, updated_message
        )

print(result.final_output)

result = await Runner.run(
    W2_Profile_Table_Agent, str(dict(result.final_output))
)
display(Markdown(result.final_output))

TaxYear=2010 Employer='Hall Ltd Group' Employer_EIN='27-0313363' Employee_Name='Daniel Robinson' Employee_SSN='522-86-4190' Employee_DOB='' Employee_DoB='' Wages_Income=91282.31 Federal_Taxes_Withheld=31479.62 Filling_Status='Single' State_Information='State code SD, Employer state ID 951-27-584, State wages 96,230.09, State income tax 3,516.46, Locality Jeffery Burgs, Local wages 96,230.09, Local tax 16,429.64' Questions='Please provide your date of birth to complete the profile.' Score='Medium'


| Field                    | Value                                                                                                                                                  |
|--------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------|
| Tax Year                 | 2010                                                                                                                                                   |
| Employer                 | Hall Ltd Group                                                                                                                                         |
| Employer EIN             | 27-0313363                                                                                                                                             |
| Employee Name            | Daniel Robinson                                                                                                                                         |
| Employee SSN             | 522-86-4190                                                                                                                                            |
| Wages Income             | 91,282.31                                                                                                                                               |
| Federal Taxes Withheld   | 31,479.62                                                                                                                                               |
| Filling Status           | Single                                                                                                                                                  |
| State Information        | State code SD, Employer state ID 951-27-584, State wages 96,230.09, State income tax 3,516.46, Locality Jeffery Burgs, Local wages 96,230.09, Local tax 16,429.64 |